# Task 4: Simple Fashion Visual Search

Compare fixed image features using cosine similarity. No pretrained model or neural training is used.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import sys
import time

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch  # Serialization only; retrieval uses NumPy on CPU.

from app.server.utils.handcrafted import DEFAULT_FEATURE_CONFIG
from app.server.utils.search_features import GARMENT_FEATURE_CONFIG
from scripts.preprocessing import SEED, task_frame
from scripts.retrieval import embed_images, retrieval_metrics

FEATURES = {
    'garment_hog_colour': dict(GARMENT_FEATURE_CONFIG),
    'pixel': {'image_size': [12, 16]},
    'hog_hsv': dict(DEFAULT_FEATURE_CONFIG),
}
for shape_weight in (0.25, 0.50, 0.75):
    FEATURES[f'hog_hsv_shape_{shape_weight:.2f}'] = {**DEFAULT_FEATURE_CONFIG, 'shape_weight': shape_weight}

def feature_type(method):
    return method if method in {'pixel', 'garment_hog_colour'} else 'hog_hsv'

(ROOT / 'models').mkdir(parents=True, exist_ok=True)
(ROOT / 'figures').mkdir(exist_ok=True)

## 2. Gallery, validation queries and test queries

Training rows form the evaluation gallery. **Validation** queries select the
feature method; test queries are evaluated only after that choice is frozen.
Selection relevance means matching both article type and catalogue base colour.
Article-only metrics are also reported. Neither is a human rating of visual similarity.
These are the existing group-isolated partitions, not a new untouched holdout.

In [ ]:
train_df = task_frame('articleType', 'train')
validation_df = task_frame('articleType', 'validation')
test_df = task_frame('articleType', 'test')
for query_frame in (validation_df, test_df):
    assert set(train_df.group_key).isdisjoint(query_frame.group_key)
assert set(validation_df.group_key).isdisjoint(test_df.group_key)
pd.Series({'gallery': len(train_df), 'validation_queries': len(validation_df),
           'test_queries': len(test_df)})

## 3. Compare fixed representations and block weightings on validation

Pixel vectors capture colour and coarse layout. HOG captures local edge patterns;
HSV histograms summarize colour. Both use bilinear resizing and unit-length
vectors. Their shared definitions are in [search_features.py](../app/server/utils/search_features.py).

The garment-aware representation adds joint HSV colour histograms, suppresses a
uniform corner-estimated background, and weights a central region to reduce
faces and trousers in modelled topwear images. This is a heuristic, not segmentation.

Keep methods within three percentage points of the original shape-weight-0.75
article precision@5, then select the highest validation article-and-colour precision@5. Exact ties use article precision@5,
then fewer feature dimensions, then the method name for deterministic ordering.
Precision@k is the matching fraction of k results; hit rate@k means at least one
match; recall@k divides matching results by **all relevant gallery items**.
MRR measures the rank of the first matching item. Query averages favor common
classes; class support and the qualitative examples matter too.

Three additional settings normalize shape and colour blocks separately and use shape weights 0.25, 0.50 and 0.75. The garment-aware candidate addresses a reported qualitative failure of the original search.
Its shape weight was tuned on the existing validation split (see
`models/visual_search_colour_evaluation.json`); this is follow-up development, not an untouched evaluation.

In [ ]:
gallery_features = {}
comparison_rows = []
for method, config in FEATURES.items():
    started = time.perf_counter()
    gallery = embed_images(train_df, feature_type(method), config)
    queries = embed_images(validation_df, feature_type(method), config)
    metrics = retrieval_metrics(queries, gallery,
                                validation_df.articleType, train_df.articleType)
    joint_metrics = retrieval_metrics(queries, gallery,
        validation_df.articleType + '|' + validation_df.baseColour,
        train_df.articleType + '|' + train_df.baseColour)
    metrics['article_and_colour_precision@5'] = joint_metrics['precision@5']
    gallery_features[method] = gallery
    comparison_rows.append({
        'method': method, **metrics, 'dimensions': gallery.shape[1],
        'gallery_mib': gallery.nbytes / 1024**2,
        'feature_and_evaluation_seconds': time.perf_counter() - started,
    })
comparison = pd.DataFrame(comparison_rows).set_index('method')
display(comparison)
minimum_article_precision = comparison.loc['hog_hsv_shape_0.75', 'precision@5'] - 0.03
comparison['eligible'] = comparison['precision@5'] >= minimum_article_precision
ranked = comparison.loc[comparison.eligible].reset_index().sort_values(
    ['article_and_colour_precision@5', 'precision@5', 'dimensions', 'method'],
    ascending=[False, False, True, True],
)
SELECTED_METHOD = ranked.iloc[0]['method']
print('Frozen method before test evaluation:', SELECTED_METHOD)
comparison.to_csv(ROOT / 'models/visual_search_validation_comparison.csv')
(ROOT / 'models/visual_search_decision.json').write_text(json.dumps({'selected': SELECTED_METHOD, 'config': FEATURES[SELECTED_METHOD], 'frozen_before_test': True}, indent=2))

## 4. Evaluate the selected method on test queries

The fixed method is evaluated once here. Do not change feature settings in response
to this table. This is a follow-up on the existing partitions; previous test
exposure must be disclosed in the final report.

In [ ]:
gallery_embeddings = gallery_features[SELECTED_METHOD]
query_embeddings = embed_images(test_df, feature_type(SELECTED_METHOD), FEATURES[SELECTED_METHOD])
test_metrics = retrieval_metrics(query_embeddings, gallery_embeddings,
                                 test_df.articleType, train_df.articleType)
pd.Series(test_metrics, name='selected_method_test')
(ROOT / 'models/visual_search_test_metrics.json').write_text(json.dumps(test_metrics, indent=2))
display(pd.Series(test_metrics, name='selected_method_test'))

## 5. Attribute agreement and examples

In [ ]:
top_five = []
for start in range(0, len(query_embeddings), 64):
    scores = query_embeddings[start:start+64] @ gallery_embeddings.T
    top_five.extend(np.argsort(-scores, axis=1, kind='stable')[:, :5])
top_five = np.asarray(top_five)
attribute_agreement = {}
for attribute in ('articleType', 'baseColour', 'gender', 'usage', 'subCategory'):
    query_values = test_df[attribute].to_numpy()[:, None]
    retrieved_values = train_df[attribute].to_numpy()[top_five]
    valid = (query_values != '') & (retrieved_values != '')
    attribute_agreement[attribute] = {
        'agreement': float((query_values == retrieved_values)[valid].mean()) if valid.any() else np.nan,
        'valid_pairs': int(valid.sum()),
    }
display(pd.DataFrame(attribute_agreement).T)
pd.DataFrame(attribute_agreement).T.to_csv(ROOT / 'models/visual_search_attribute_agreement.csv')
per_query = pd.DataFrame({'id': test_df.id.to_numpy(), 'articleType': test_df.articleType.to_numpy(), 'precision@5': (train_df.articleType.to_numpy()[top_five] == test_df.articleType.to_numpy()[:, None]).mean(1)})
per_class = per_query.groupby('articleType').agg(queries=('id','size'), precision_at_5=('precision@5','mean'))
display(per_class.sort_values('precision_at_5').head(15))
per_class.to_csv(ROOT / 'models/visual_search_per_class.csv')
np.save(ROOT / 'models/visual_search_top_five.npy', top_five)

Attribute agreement is a secondary diagnostic; literal `NA` counts as a usage
label, while genuinely blank pairs are excluded and their support is reported.
The fixed panel below includes common and rare article types plus catalogue
examples. It is descriptive, not a curated demonstration of only good matches.

In [ ]:
support = train_df.articleType.value_counts()
common = test_df.loc[test_df.articleType.isin(support.head(5).index)].head(2)
rare = test_df.loc[test_df.articleType.map(support).le(49)].head(2)
panel = pd.concat([common, rare, test_df.sample(6, random_state=SEED)])
query_indices = panel.drop_duplicates('id').head(6).index
positions = test_df.index.get_indexer(query_indices)
fig, axes = plt.subplots(len(positions), 6, figsize=(12, 2.5 * len(positions)), squeeze=False)
for row, position in enumerate(positions):
    items = [test_df.iloc[position]]
    items.extend(train_df.iloc[index] for index in top_five[position])
    for column, item in enumerate(items):
        with Image.open(item.image_path) as image:
            axes[row, column].imshow(image.convert('RGB'))
        axes[row, column].set_title(('Query: ' if column == 0 else '') + item.articleType, fontsize=8)
        axes[row, column].axis('off')
plt.tight_layout()
fig.savefig(ROOT / 'figures/retrieval_examples.png', dpi=130, bbox_inches='tight')
figures = ROOT / 'figures'
figures.mkdir(exist_ok=True)
fig.savefig(figures / 'retrieval_examples.png', dpi=220, bbox_inches='tight')

## 6. Export the deployment gallery

Each run replaces the search checkpoint, embeddings, metadata and history directly in `models/` after validation selection and internal-test evaluation.

In [ ]:
deployment_gallery = pd.concat([train_df, validation_df, test_df], ignore_index=True)
deployment_gallery = deployment_gallery.drop_duplicates('id')
deployment_embeddings = embed_images(deployment_gallery, feature_type(SELECTED_METHOD), FEATURES[SELECTED_METHOD])
checkpoint = {
    'model_type': 'fixed_feature_cosine', 'feature_type': feature_type(SELECTED_METHOD), 'selected_method': SELECTED_METHOD,
    'feature_config': FEATURES[SELECTED_METHOD],
    'embedding_dim': deployment_embeddings.shape[1],
    'selection_metric': 'validation_article_and_colour_precision@5',
    'validation_metrics': comparison.loc[SELECTED_METHOD].to_dict(),
    'retrieval_metrics': test_metrics, 'seed': SEED,
    'experiment': 'garment_colour_search_v2',
}
model_dir = ROOT / 'models'
model_dir.mkdir(parents=True, exist_ok=True)
np.save(model_dir / 'visual_search_embeddings.npy', deployment_embeddings)
deployment_gallery[['id', 'image_path', 'articleType', 'baseColour', 'gender', 'usage', 'subCategory']].to_csv(
    model_dir / 'visual_search_metadata.csv', index=False,
)
torch.save(checkpoint, model_dir / 'visual_search_model.pt')
comparison.to_csv(model_dir / 'visual_search_history.csv')
print('Deployment index MiB:', deployment_embeddings.nbytes / 1024**2)
print('Replaced the deployment model, embeddings, metadata and history in models/.')